In [ ]:
print("Hello, World!")

In [ ]:
%pip install accelerate datasets evaluate rouge_score bert_score sacrebleu -q

In [ ]:
%pip install --upgrade transformers trl -q

In [ ]:
# %%
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import DPOTrainer, DPOConfig
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import evaluate

In [ ]:
# %%
# --- 2. Configuration ---
MODEL_NAME = "gpt2"
DATASET_NAME = "HumanLLMs/Human-Like-DPO-Dataset"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
EPOCHS = 1
LEARNING_RATE = 5e-6
MAX_LENGTH = 512
MAX_RESPONSE_LENGTH = 256
EXP_NAME = "dpop_human_alignment" # Changed from ppo to dpo

BASE_DIR = f"runs/{EXP_NAME}"
CKPT_DIR = f"{BASE_DIR}/checkpoints"
LOG_DIR = f"{BASE_DIR}/logs"
PLOT_DIR = f"{BASE_DIR}/plots"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

In [ ]:
# %%
# --- 3. Load Model & Data ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
dataset = load_dataset(DATASET_NAME, split="train").train_test_split(test_size=0.2)

# Subsetting for efficiency
train_dataset = dataset["train"].select(range(min(len(dataset["train"]), 2048)))
eval_dataset = dataset["test"].select(range(min(len(dataset["test"]), 512)))

In [ ]:
def fix_format(example):
    example["prompt"] = example["prompt"].strip()
    example["chosen"] = " " + example["chosen"].strip()
    example["rejected"] = " " + example["rejected"].strip()
    return example

train_dataset = train_dataset.map(fix_format)
eval_dataset = eval_dataset.map(fix_format)

In [ ]:
# %%
# --- 4. Define Generative Metrics (Local) ---
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def compute_generative_metrics(model, tokenizer, test_data):
    model.eval()
    predictions = []
    references = []
    
    print(f"Generating responses for {len(test_data)} samples...")
    for example in test_data:
        prompt = example["prompt"]
        target = example["chosen"]
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=64, 
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                repetition_penalty=1.2,
                no_repeat_ngram_size=2
            )
        
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):].strip()
        
        # --- CRITICAL FIX START ---
        # BERTScore crashes on empty strings. If pred_text is empty, use a placeholder.
        if not pred_text or pred_text.strip() == "":
            pred_text = "." 
        # --- CRITICAL FIX END ---
        
        predictions.append(pred_text)
        references.append(target)

    # Compute Scores
    bleu = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    rouge = rouge_metric.compute(predictions=predictions, references=references)
    
    # We add model_type="roberta-large" and use_fast_tokenizer=False to be extra safe
    bert = bertscore_metric.compute(
        predictions=predictions, 
        references=references, 
        lang="en", 
        model_type="roberta-large"
    )

    return {
        "bleu": bleu["score"],
        "rougeL": rouge["rougeL"],
        "bert_f1": np.mean(bert["f1"])
    }, predictions

In [ ]:
# %%
def compute_metrics(eval_output):
    """Calculates Accuracy/F1 based on DPOP rewards (Chosen vs Rejected)."""
    # predictions can sometimes be a tuple (rewards, logits, etc.) 
    # and might be flattened or 2D depending on the TRL version.
    preds = eval_output.predictions
    
    # If preds is a tuple, we usually want the first element (the rewards/logits)
    if isinstance(preds, tuple):
        preds = preds[0]
        
    # If the array is 1D, TRL has flattened the [chosen_rewards, rejected_rewards]
    # We need to reshape it back to (N, 2)
    if len(preds.shape) == 1:
        # TRL often concatenates them, so we split them in half
        half = len(preds) // 2
        rewards_chosen = preds[:half]
        rewards_rejected = preds[half:]
    else:
        # Standard (N, 2) case
        rewards_chosen = preds[:, 0]
        rewards_rejected = preds[:, 1]
    
    # Accuracy: % of times chosen reward > rejected reward
    predictions = (rewards_chosen > rewards_rejected).astype(int)
    labels = np.ones_like(predictions) # Ground truth is always 1
    
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary', zero_division=0
    )
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,   
        "reward_margin": (rewards_chosen - rewards_rejected).mean()
    }

In [ ]:
from copy import deepcopy

# ---- reference model (frozen SFT baseline) ----
ref_model = deepcopy(model)
ref_model.eval()

for p in ref_model.parameters():
    p.requires_grad = False

training_args = DPOConfig(
    output_dir=CKPT_DIR,
    report_to="tensorboard", 
    logging_dir=LOG_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    
    # --- MEMORY FIXES START ---
    eval_strategy="steps",
    eval_steps=20,
    per_device_eval_batch_size=1,       # Reduce eval batch size to the absolute minimum
    eval_accumulation_steps=1,          # Move tensors to CPU faster during eval
    # ---------------------------
    
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    remove_unused_columns=False,
    save_total_limit=2,
    loss_type="discopop",
    beta=0.005,
    max_length=MAX_LENGTH,
    max_steps=300,
    fp16=True if torch.cuda.is_available() else False,
    gradient_checkpointing=True,
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,   # ← REQUIRED FIX
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# %%
# --- 7. Execution ---
print("Starting DPOP training...")
trainer.train()

In [ ]:
# %%
# --- 8. Final Evaluation & Local Plotting ---
print("\n--- Running Final Generative Evaluation ---")
gen_metrics, preds = compute_generative_metrics(model, tokenizer, eval_dataset)

# Save Generative Metrics to a local file
with open(f"{BASE_DIR}/final_metrics.txt", "w") as f:
    for k, v in gen_metrics.items():
        f.write(f"{k.upper()}: {v:.4f}\n")
        print(f"{k.upper()}: {v:.4f}")

In [ ]:
# %%
def plot_local_results(trainer):
    """Parses trainer logs and saves plots to PLOT_DIR."""
    history = pd.DataFrame(trainer.state.log_history)
    
    # 1. Plot Loss
    plt.figure(figsize=(10, 5))
    train_loss = history[history['loss'].notna()]
    plt.plot(train_loss['step'], train_loss['loss'], label='Train Loss')
    plt.title("DPOP Training Loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(f"{PLOT_DIR}/loss_curve.png")
    plt.show()

    # 2. Plot Reward Accuracy & Margin
    eval_logs = history[history['eval_accuracy'].notna()]
    if not eval_logs.empty:
        fig, ax1 = plt.subplots(figsize=(10, 5))
        
        ax1.set_xlabel('Step')
        ax1.set_ylabel('Accuracy / F1', color='tab:blue')
        l1, = ax1.plot(eval_logs['step'], eval_logs['eval_accuracy'], label='Accuracy', color='tab:blue', marker='o')
        l2, = ax1.plot(eval_logs['step'], eval_logs['eval_f1'], label='F1', color='tab:cyan', linestyle='--')
        ax1.tick_params(axis='y', labelcolor='tab:blue')

        ax2 = ax1.twinx()
        ax2.set_ylabel('Reward Margin', color='tab:red')
        l3, = ax2.plot(eval_logs['step'], eval_logs['eval_reward_margin'], label='Margin', color='tab:red', marker='x')
        ax2.tick_params(axis='y', labelcolor='tab:red')

        # Combine legends from both axes
        lines = [l1, l2, l3]
        labels = [line.get_label() for line in lines]
        ax1.legend(lines, labels, loc='best')

        plt.title("Preference Learning Progress")
        fig.tight_layout()
        plt.savefig(f"{PLOT_DIR}/metrics_curve.png")
        plt.show()

plot_local_results(trainer)

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
print(history.columns)

In [ ]:
# Save side-by-side comparison to CSV for local review
comparison_df = pd.DataFrame({
    "Prompt": [eval_dataset[i]["prompt"] for i in range(len(preds))],
    "Target (Human)": [eval_dataset[i]["chosen"] for i in range(len(preds))],
    "Model Output": preds
})
comparison_df.to_csv(f"{BASE_DIR}/generations_comparison.csv", index=False)
print(f"Evaluation complete. Results saved to {BASE_DIR}")

In [ ]:
formatted_prompt = f"Question: Can you bake a cake?\n\nAnswer: "
        
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

# 2. Generation with Diversity Constraints (prevents model collapse)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2,      # Crucial to stop loops
        no_repeat_ngram_size=2,       # Prevents phrase stuttering
        pad_token_id=tokenizer.eos_token_id
    )

# 3. Decode and Clean Output
full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract only the model's generated part (after the prompt)
response = full_text[len(formatted_prompt):].strip()

print(f"{formatted_prompt}{response}")